# Day 25: NLP Basics and a Classical Chatbot

On Day 24 an **LLM** read the tea shop messages for us. Today we do the same job the **old way**: with the classical ML from Week 4. No API, no key.

## 🎯 The example we will build

The same **tea shop**. A customer writes a message. Our bot finds the **intent** (what the customer wants) and sends a **ready reply**.

| Customer writes | Intent | Bot replies |
|---|---|---|
| *Hi there* | `greeting` | Assalam o Alaikum! Welcome to Chai Wala. How can I help? |
| *What is on the menu* | `menu` | We have chai, paratha, samosa and cake rusk. |
| *Do you deliver to Gulberg* | `delivery` | We deliver in Lahore from 9 am to 11 pm. Delivery is Rs 100. |
| *The chai was cold* | `complaint` | Sorry for the trouble! Please send your order number. |

At the end we **break** our bot, and see why LLMs were a big step.

## 📋 Agenda

We climb from the bottom. Each step uses the step below.

```
TOP     Step 8   Old way vs LLM           what we learned
          ▲
        Step 7   Where it breaks          test hard messages
          ▲
        Step 6   The chatbot              intent → ready reply
          ▲
        Step 5   Train the model          Logistic Regression (Week 4)
          ▲
        Step 4   TF-IDF                   important words get big numbers
          ▲
        Step 3   Bag of words             count the words
          ▲
        Step 2   Tokenize                 cut text into words
          ▲
BOTTOM  Step 1   Load the data            40 messages with their intent
```

### New words today

| Word | Meaning |
|---|---|
| **NLP** | Natural Language Processing: computers working with human text. |
| **Intent** | What the customer wants: greeting, menu, delivery or complaint. |
| **Token** | One small piece of text. Today: one word. |
| **Tokenize** | Cut a sentence into tokens. |
| **Vocabulary** | The list of all different words the model knows. |
| **Bag of words** | Turn a sentence into numbers by **counting** each word. Order is lost. |
| **TF-IDF** | Like bag of words, but rare (important) words get bigger numbers. |
| **Canned reply** | A ready reply, written by us before. The bot only picks one. |

---
# Class Problem: a chatbot for the tea shop

## Step 1: Load the data

- A model learns from **examples**. Here: 40 messages, each with its **intent**.
- 10 messages for each of the 4 intents.
- `text` is the input (X). `intent` is the answer (y). Same as Week 4.

In [ ]:
import pandas as pd

df = pd.read_csv("tea_shop_intents.csv")
print(df.shape)                       # (40, 2): 40 messages, 2 columns
df.head()

In [ ]:
df["intent"].value_counts()           # 10 messages for each intent

## Step 2: Tokenize

- A model needs **numbers**. Text is not numbers.
- First we cut the text into **words** (tokens).
- We make all letters **small**, so `Chai` and `chai` are the same word.
- We remove `?` `!` `.` `,` so `chai?` and `chai` are the same word.

In [ ]:
import string

def tokenize(text):
    text = text.lower()                                               # Chai → chai
    text = text.translate(str.maketrans("", "", string.punctuation))  # remove ? ! . ,
    return text.split()                                               # cut at spaces

print(tokenize("The chai was COLD!"))     # ['the', 'chai', 'was', 'cold']

**✏️ Quick Check:** Tokenize `"Do you DELIVER to Gulberg?"`. What do you get?

In [ ]:
# your code here

**Answer:** all small letters, no `?`, 5 words.

In [ ]:
print(tokenize("Do you DELIVER to Gulberg?"))     # ['do', 'you', 'deliver', 'to', 'gulberg']

## Step 3: Bag of words

- Make a **vocabulary**: every different word, in A–Z order.
- For each sentence, **count** how many times each word comes.
- The sentence is now a list of numbers. The model can read it.

*Everyday example:* you empty a shopping bag on the table and count: 2 eggs, 1 bread. You do not know the **order** they went in. Bag of words is the same: it counts words and forgets the order.

**By hand, with 2 sentences:**

| Sentence | chai | cold | good | is | the |
|---|---|---|---|---|---|
| the chai is cold | 1 | 1 | 0 | 1 | 1 |
| the chai is good | 1 | 0 | 1 | 1 | 1 |

Only **one number** is different: that is how the model sees *cold* vs *good*.

`CountVectorizer` from scikit-learn does this for us. It also does Step 2 (small letters, remove punctuation).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

two = ["the chai is cold", "the chai is good"]

bow = CountVectorizer()
counts = bow.fit_transform(two)            # fit = make the vocabulary, transform = count

print(bow.get_feature_names_out())         # the vocabulary
print(counts.toarray())                    # one row per sentence, same as our table

**✏️ Quick Check:** With the same `bow`, what are the numbers for `"chai chai cold"`? Use `bow.transform([...])`.

In [ ]:
# your code here

**Answer:** `chai` is counted 2 times. `good`, `is`, `the` are 0.

In [ ]:
print(bow.transform(["chai chai cold"]).toarray())     # [[2 1 0 0 0]]

## Step 4: TF-IDF

**Problem with counting:** words like `the`, `is`, `you` come in almost every message. They do not tell us the intent. But they get the same count as `cold` or `menu`.

**TF-IDF fixes this:**

- **TF** (term frequency): how many times the word is in **this** message.
- **IDF** (inverse document frequency): how **rare** the word is in **all** messages.
- TF-IDF = TF × IDF. A **rare** word gets a **big** number. A word in many messages gets a **small** number.

*Everyday example:* in a class, everyone is called "student". That name does not help you find Ali. His **roll number** is rare, so it helps.

Now we use all 40 messages.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["text"])     # learn the vocabulary + IDF from all messages
y = df["intent"]

print(X.shape)                               # (40 messages, number of words in the vocabulary)

In [ ]:
# The numbers for one message. Bigger number = more important word.
row = vectorizer.transform(["The chai was cold"]).toarray()[0]
words = vectorizer.get_feature_names_out()

pd.Series(row, index=words)[row > 0].sort_values(ascending=False)

**✏️ Quick Check:** In `"What is the delivery charge"`, which word gets the **biggest** number, and which gets the **smallest**?

In [ ]:
# your code here

**Answer:** `charge` is the biggest: it is in only 1 message. `the` and `is` are small: they are in many messages.

In [ ]:
row = vectorizer.transform(["What is the delivery charge"]).toarray()[0]
pd.Series(row, index=words)[row > 0].sort_values(ascending=False)

## Step 5: Train the model

- Now it is the same as Week 4: **X** (numbers) and **y** (labels).
- We use **Logistic Regression** (Day 17). It learns which words go with which intent.
- We do **not** split into train and test today. 40 messages is very few, so we train on all of them and test with **new messages we write ourselves**.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

new = ["Do you deliver to Johar Town"]
print(model.predict(vectorizer.transform(new)))     # ['delivery']

**Important:** a new message must go through the **same** `vectorizer` with `transform` (not `fit_transform`). The model only knows the vocabulary it learned in Step 4.

**✏️ Quick Check:** Predict the intent of `"Can I see the menu please"`.

In [ ]:
# your code here

**Answer:** `menu`. The word `menu` is in the vocabulary, and the model learned it goes with `menu`.

In [ ]:
print(model.predict(vectorizer.transform(["Can I see the menu please"])))     # ['menu']

## Step 6: The chatbot

- For each intent, we write **one ready reply** (a canned reply) in a dictionary.
- The bot: message → TF-IDF numbers → model finds the intent → reply from the dictionary.
- The model does **not write** text. It only **picks a box**.

In [ ]:
REPLIES = {
    "greeting":  "Assalam o Alaikum! Welcome to Chai Wala. How can I help?",
    "menu":      "We have chai, paratha, samosa and cake rusk.",
    "delivery":  "We deliver in Lahore from 9 am to 11 pm. Delivery is Rs 100.",
    "complaint": "Sorry for the trouble! Please send your order number.",
}

def chatbot(message):
    x = vectorizer.transform([message])       # text → numbers
    intent = model.predict(x)[0]              # numbers → intent
    return intent, REPLIES[intent]            # intent → ready reply

for m in ["Hi there", "What is on the menu", "Do you deliver to DHA", "The chai was cold"]:
    intent, reply = chatbot(m)
    print(f"{m:25} → {intent:10} → {reply}")

## Step 7: Where it breaks

The bot works on messages that look like the training data. Now we try real customer messages.

In [ ]:
hard = [
    "delivery charges?",                     # 1. normal message
    "dilivery charges?",                     # 2. same message, one spelling mistake
    "Is the tea hot?",                       # 3. 'tea' - our data only says 'chai'
    "What is the capital of Pakistan",       # 4. not about the tea shop at all
    "Send 3 chai to Gulberg. Last time it came cold",   # 5. two things in one message
]

for m in hard:
    intent, reply = chatbot(m)
    print(f"{m:48} → {intent:10} → {reply}")

**What went wrong?**

| # | Message | Problem |
|---|---|---|
| 2 | *dilivery charges?* | `dilivery` is not in the vocabulary. All numbers are 0. The model guesses. |
| 3 | *Is the tea hot?* | `tea` is a new word. The model does not know `tea` = `chai`. |
| 4 | *capital of Pakistan* | The bot **must** pick one of the 4 intents. It cannot say "I don't know". |
| 5 | *3 chai to Gulberg, came cold* | Only **one** intent. The reply ignores "3 chai" and "Gulberg". |

We can see problem 2 in the numbers:

In [ ]:
print(vectorizer.transform(["delivery charges?"]).sum())    # some numbers are not 0
print(vectorizer.transform(["dilivery charges?"]).sum())    # 0.0: the bot sees NOTHING

### A small patch

If the bot sees **no known words**, say sorry. This fixes message 2, but only with a "sorry". It does not fix 3, 4 or 5.

In [ ]:
def chatbot(message):
    x = vectorizer.transform([message])
    if x.sum() == 0:                                    # no word is in the vocabulary
        return "unknown", "Sorry, I did not understand. Please write again."
    intent = model.predict(x)[0]
    return intent, REPLIES[intent]

print(chatbot("dilivery charges?"))

**✏️ Quick Check:** Send `"Thank you so much"` to the chatbot. Is the answer right? Why?

In [ ]:
# your code here

**Answer:** No. It says `menu`. There is no `thanks` intent in our data, and `you` is a known word, so the bot picks a box anyway. To fix it, we must add a new intent, new examples and a new reply by hand, and train again.

In [ ]:
print(chatbot("Thank you so much"))     # ('menu', ...): wrong

## Step 8: Old way vs LLM

| | Our bot (today) | LLM bot (Day 24) |
|---|---|---|
| Training data | We write it by hand | Already trained on a lot of text |
| Spelling mistake | Breaks | Understands |
| New word (`tea`) | Breaks | Understands |
| Not about the shop | Picks a wrong intent | Can say "I only help with the tea shop" |
| Reply | Same ready text every time | Writes a new reply with details (3 chai, Gulberg) |
| Cost and speed | Free, very fast, runs on your laptop | Needs an API, costs tokens, slower |
| Control | We know every reply | It can say something wrong |

**So the old way is not useless:** it is free, fast, and safe. But it is **rigid**: it only knows what we taught it.

## ✅ Complete code (Class Problem)

All the steps in one cell. You can run this cell alone.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Step 1: load the data
df = pd.read_csv("tea_shop_intents.csv")

# Steps 2-4: tokenize + TF-IDF numbers
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["text"])
y = df["intent"]

# Step 5: train the model
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# Step 6: ready replies + chatbot (with the patch from Step 7)
REPLIES = {
    "greeting":  "Assalam o Alaikum! Welcome to Chai Wala. How can I help?",
    "menu":      "We have chai, paratha, samosa and cake rusk.",
    "delivery":  "We deliver in Lahore from 9 am to 11 pm. Delivery is Rs 100.",
    "complaint": "Sorry for the trouble! Please send your order number.",
}

def chatbot(message):
    x = vectorizer.transform([message])
    if x.sum() == 0:
        return "unknown", "Sorry, I did not understand. Please write again."
    intent = model.predict(x)[0]
    return intent, REPLIES[intent]

# Try it
for m in ["Hi there", "Do you deliver to DHA", "The chai was cold", "dilivery charges?"]:
    print(m, "→", chatbot(m))

### Key takeaways

- **NLP** = computers working with text. A model needs **numbers**, so we turn text into numbers first.
- **Tokenize** = cut text into words (small letters, no punctuation).
- **Bag of words** = count each word. The **order is lost**.
- **TF-IDF** = rare words get big numbers, common words (`the`, `is`) get small numbers.
- A **classical chatbot** = find the intent, then send a **ready reply**.
- It breaks on **spelling mistakes**, **new words**, **other topics** and **long messages**. It only knows what we taught it.
- An **LLM** handles these, but it costs money and is less controlled.

---
# 🏠 Home Problem: the tea shop adds 2 new intents

Customers also ask about **shop timings** and **how to pay**. Open `tea_shop_intents_home.csv`. It has **6 intents**: the old 4, plus `hours` and `payment`.

Do the **same steps**:

1. Load `tea_shop_intents_home.csv`. Check how many messages each intent has.
2. Make TF-IDF numbers with `TfidfVectorizer`.
3. Train a `LogisticRegression` model.
4. Write a `REPLIES` dictionary with **6** ready replies. Example for `hours`: *"We are open every day from 9 am to 11 pm."*
5. Write the `chatbot` function (with the "no known words" patch).
6. Test 5 messages: `"When do you close"`, `"Can I pay by JazzCash"`, `"Do you take easypaisa"`, `"Are you open on Eid"`, `"Can I pay when the rider comes"`.
7. In 2 lines: is every **intent** right? Does every **reply** really answer the customer?